In [3]:
#234567890#234567890#234567890#234567890#234567890#234567890#234567890#23456789
from copy import deepcopy

from bertopic import BERTopic
from bertopic.representation import (
    KeyBERTInspired, MaximalMarginalRelevance, OpenAI, TextGeneration)
from datasets import load_dataset
from hdbscan import HDBSCAN
import matplotlib.pyplot as plt
import numpy as np
import openai
import pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from umap import UMAP

In [2]:
dataset = load_dataset('maartengr/arxiv_nlp')['train']
abstracts = dataset['Abstracts']
titles = dataset['Titles']

In [ ]:
embedding_mod = SentenceTransformer('thenlper/gte-small')
embeddings = embedding_mod.encode(abstracts, show_progress_bar=True)

Batches:   0%|          | 0/1405 [00:00<?, ?it/s]

In [ ]:
embeddings.shape

In [ ]:
umap_mod = UMAP(
    n_components=5, min_dist=0., metric='cosine', random_state=123)
reduced_embeddings = umap_mod.fit_transform(embeddings)

In [ ]:
hdbscan_mod = HDBSCAN(
    min_cluster_size=50, metric='euclidean', cluster_selection_method='eom'
).fit(reduced_embeddings)
clusters = hdbscan_mod.labels_
len(set(clusters))

### Inspecting Clusters

In [ ]:
cluster = 1
for i in np.where(clusters == cluster)[0][:3]:
    print(abstracts[i][:300])
    print()

In [ ]:
reduced_embeddings = UMAP(
    n_components=2, min_dist=0., metric='cosine', random_state=123
).fit_transform(embeddings)

In [ ]:
df = pd.DataFrame(reduced_embeddings, columns=['x', 'y'])
df['title'] = titles
df['cluster'] = [str(c) for c in clusters]

In [ ]:
to_plot = df.loc[df.cluster != '-1', :]
outliers = df.loc[df.cluster == '-1', :]

In [ ]:
plt.scatter(outliers.x, outliers.y, alpha=0.05, s=2, c='grey')
plt.scatter(
    to_plot.x,
    to_plot.y, 
    c=to_plot.cluster.astype(int),
    alpha=0.6,
    s=2,
    cmap='tab20b')
plt.axis('off');

## From Text Clustering to Topic Modeling

In [ ]:
topic_mod = BERTopic(
    embedding_model=embedding_mod,
    umap_model=umap_mod,
    hdbscan_model=hdbscan_mod,
    verbose=True
).fit(asbtracts, embeddings)

In [ ]:
topic_mod.get_topic_info()

In [ ]:
topic_mod.get_topic(0)

In [ ]:
topic_mod.find_topics('topic modeling')

In [ ]:
topic_mod.get_topic(22)

In [ ]:
topic_mod.topics_[
    titles.index(
        'BERTopic: Neural topic modeling with a class-baed TF-IDF procedure')]

In [ ]:
fig = topic_mod.visualize_documents(
    titles, 
    reduced_embeddings=reduced_embeddings, 
    width=1200, 
    hide_annotations=True)

In [ ]:
fig.update_layout(font={'size': 16})

In [ ]:
topic_mod.visualize_barchart()

In [ ]:
topic_mod.visualize_heatmap(n_cluster=30)

In [ ]:
topic_mod.viusalize_hierarchy()

In [ ]:
original_topics = deepcopy(topic_model.topic_representations_)

In [ ]:
def topic_differences(mod, orig_topics, n_topics=5):
    'Show the differences in topic representations between 2 models'
    df = pd.DataFrame(columns=['Topic', 'Original', 'Updated'])
    for i in range(n_topics):
        orig_words = '|'.join(list(zip(*orig_topics[i]))[0][:5])
        new_words = '|'.join(list(zip(*mod.get_topic(i)))[0][:5])
        df.loc[len(df)] = [topic, orig_words, new_words]
    return df

In [ ]:
representation_mod = KeyBERTInspired()
topic_mod.update_topics(abstracts, representation_model=representation_mod)
topic_differences(topic_mod, original_topics)

In [ ]:
representation_mod = MaximalMarginalRelevance(diversity=0.2)
topic_mod.update_topics(abstracts, representation_model=representation_mod)
topic_differences(topic_mod, original_topics)

In [ ]:
prompt = (
    '''I have a topic that contains the following documnets:
    [DOCUMENTS]

    The topic is described by the following keywords: '[KEYWORDS]'.

    Based on the documents and keywords, what is this topic about?''')

In [ ]:
genearator = pipeline('text2text-generation', model='google/flan-t5-small')
representation_mod = TextGeneration(
    generator, prompt=prompt, doc_length=50, tokenizer='whitespace')
topic_model.update_topics(abstracts, representation_model=representation_mod)
topic_differences(topic_model, original_topics)

In [ ]:
prompt = (
    '''I have a topic that contains the following documnets:
    [DOCUMENTS]

    The topic is described by the following keywords: '[KEYWORDS]'.

    Based on the information above, exract a short topic label n the following
    fromat topic: <short topic label>''')

In [ ]:
client = openai.OpenAI(api_key='MYKEY')
representation_mod = OpenAI(
    client, 
    model='gpt-3.5-turbo', 
    exponential_backoff=True, 
    chat=True, 
    prompt=prompt)
topic_model.update_topics(abstracts, representation_model=representation_mod)
topic_differences(topic_model, original_topics)